# 08 — ASR and Content Representation

## Objective

Prepare the **original/anonymized speech pairs** required for content-preservation evaluation.

The anonymization method is intentionally left open. Whatever route is chosen later must produce an anonymized audio file paired with its corresponding original audio.

$$
(X_o,X_a)\xrightarrow{\mathrm{ASR}}(W_o,W_a)
\xrightarrow{E}(Z_o,Z_a)
$$

The transcript pair $(W_o,W_a)$ is then passed to:

$$
(W_o,W_a)\rightarrow
\begin{cases}
\text{Semantic Similarity}\\
\text{NLI}\\
\text{Semantic alignment}\rightarrow M\rightarrow\text{TPI}
\end{cases}
$$

**Principles:** same ASR system/configuration on both sides; preserve raw ASR transcripts; do not choose an anonymization route here; do not combine metrics into a final loss here.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

from faster_whisper import WhisperModel

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Results directory:", RESULTS_DIR)

Project root: /Users/juni/Documents/interview-anonymization
Results directory: /Users/juni/Documents/interview-anonymization/data/results


/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. ASR configuration

Faster-Whisper is used to obtain $W_o$ and $W_a$.

The **same model and decoding configuration** must be used for original and anonymized audio so that transcript differences are not caused by changing the ASR system.

In [2]:
ASR_MODEL_NAME = "base"
ASR_DEVICE = "cpu"
ASR_COMPUTE_TYPE = "int8"
ASR_LANGUAGE = "en"

asr_model = WhisperModel(
    ASR_MODEL_NAME,
    device=ASR_DEVICE,
    compute_type=ASR_COMPUTE_TYPE,
)

print("ASR model:", ASR_MODEL_NAME)
print("Device:", ASR_DEVICE)
print("Language:", ASR_LANGUAGE)


ASR model: base
Device: cpu
Language: en


## 2. Content encoder

The encoder converts each ASR transcript into a semantic representation.

Keep the model selected in the dedicated model-selection notebook and use the **same encoder for both sides**.

These embeddings support semantic similarity. They are not a replacement for NLI or TPI.

## 3. ASR helper

Keep the transcript as produced by ASR. Do not apply semantic corrections or aggressive normalization at this stage.

In [3]:
def transcribe_audio(audio_path, model=asr_model, language=ASR_LANGUAGE):
    """Transcribe one audio file with the fixed ASR configuration."""
    audio_path = Path(audio_path)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    segments, info = model.transcribe(
        str(audio_path),
        language=language,
    )

    return " ".join(
        segment.text.strip()
        for segment in segments
        if segment.text.strip()
    )


## 4. ASR sanity check

VoxCeleb is used here only to verify that the ASR pipeline works. This is **not yet** the content-preservation experiment, because that experiment requires an original/anonymized pair.

In [4]:
utterance_df = pd.read_parquet(
    METADATA_DIR / "voxceleb_utterances.parquet"
)

utterance_df["path"] = utterance_df["path"].str.replace(
    "/Users/juni/Documents/voxceleb1/",
    "/Users/juni/Documents/interview-anonymization/voxceleb1/",
    regex=False
)

wav_files = [Path(p) for p in utterance_df["path"]]

print("Number of wav files:", len(wav_files))

sample_path = wav_files[0]
print("Sample:", sample_path)
print("Exists:", sample_path.exists())

Number of wav files: 124843
Sample: /Users/juni/Documents/interview-anonymization/voxceleb1/wav/id10001/1zcIwhmdeo4/00001.wav
Exists: True


In [5]:
print("Number of utterances:", len(utterance_df))
print("Number of speakers:", utterance_df["speaker_id"].nunique())

sample = utterance_df.iloc[0]
audio_path = Path(sample["path"])

print("Speaker:", sample["speaker_id"])
print("Video:", sample["video_id"])
print("Split:", sample["split"])
print("Language:", sample["language"])
print("Audio:", audio_path)
print("Exists:", audio_path.exists())


Number of utterances: 124843
Number of speakers: 1166
Speaker: id10001
Video: 1zcIwhmdeo4
Split: train
Language: english
Audio: /Users/juni/Documents/interview-anonymization/voxceleb1/wav/id10001/1zcIwhmdeo4/00001.wav
Exists: True


In [6]:
sample_transcription = transcribe_audio(audio_path)

print("ASR transcription:")
print(sample_transcription)


ASR transcription:
Looking guys up that lived in the Everglades and stuff and and that my character was described as a romantic sociopath who was good with a knife so I figured


## 5. Pair-level data interface

The real experiment needs a table like:

| pair_id | original_path | anonymized_path |
|---|---|---|
| ... | original audio | anonymized audio |

The anonymization route is **not specified here**. Once a route is selected, its outputs are registered as `anonymized_path` for the corresponding `pair_id`.

Additional metadata such as speaker, split and dataset can be retained.

In [7]:
# Schema required by the ASR pipeline.
pair_columns = ["pair_id", "original_path", "anonymized_path"]

# Example only:
# pairs_df = pd.DataFrame({
#     "pair_id": ["001", "002"],
#     "original_path": ["...", "..."],
#     "anonymized_path": ["...", "..."],
# })

print("Required columns:", pair_columns)


Required columns: ['pair_id', 'original_path', 'anonymized_path']


## 6. Transcribe both sides

For each pair:

$$
X_o\rightarrow W_o,
\qquad
X_a\rightarrow W_a.
$$

The paired transcripts are the main output of this notebook and feed Semantic Similarity, NLI, semantic alignment and TPI.

For the first real experiment, process a small subset (e.g. 100 pairs) before scaling up.

In [8]:
def transcribe_pairs(
    pairs_df,
    original_column="original_path",
    anonymized_column="anonymized_path",
):
    """Transcribe both audio files for every original/anonymized pair."""
    rows = []

    for _, row in pairs_df.iterrows():
        original_transcript = transcribe_audio(row[original_column])
        anonymized_transcript = transcribe_audio(row[anonymized_column])

        result = row.to_dict()
        result["original_transcript"] = original_transcript
        result["anonymized_transcript"] = anonymized_transcript
        rows.append(result)

    return pd.DataFrame(rows)


## 7. Encode original and anonymized transcripts

Use the same encoder:

$$
Z_o=E(W_o),
\qquad
Z_a=E(W_a).
$$

The embeddings can later be compared with cosine similarity.

A high semantic similarity does **not** prove that no information was deleted; this is why NLI and TPI remain separate.

In [9]:
def encode_transcript_pairs(
    transcript_df,
    original_column="original_transcript",
    anonymized_column="anonymized_transcript",
):
    """Encode original and anonymized transcripts with the same encoder."""
    original_embeddings = content_encoder.encode(
        transcript_df[original_column].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    anonymized_embeddings = content_encoder.encode(
        transcript_df[anonymized_column].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    return original_embeddings, anonymized_embeddings


## 8. End-to-end pair pipeline

Once `pairs_df` exists:

$$
\boxed{
(X_o,X_a)
\rightarrow
(W_o,W_a)
\rightarrow
(Z_o,Z_a)
}
$$

The text pair $(W_o,W_a)$ is then passed unchanged to the content-evaluation stage.

In [10]:
# Run once pairs_df has been created.

# transcript_df = transcribe_pairs(pairs_df)
#
# original_embeddings, anonymized_embeddings = encode_transcript_pairs(
#     transcript_df
# )
#
# transcript_df["original_embedding"] = list(original_embeddings)
# transcript_df["anonymized_embedding"] = list(anonymized_embeddings)
#
# transcript_df.to_pickle(
#     RESULTS_DIR / "content_pairs_asr_embeddings.pkl"
# )
#
# print("Pairs processed:", len(transcript_df))
# print("Embedding shape:", original_embeddings.shape)


## 9. Output for content evaluation

For each pair we retain:

- original audio path;
- anonymized audio path;
- original transcript $W_o$;
- anonymized transcript $W_a$;
- original embedding $Z_o$;
- anonymized embedding $Z_a$.

The important interface is:

$$
\boxed{
(W_o,W_a)
\rightarrow
\begin{cases}
\text{Semantic Similarity}\\
\text{NLI}\\
\text{Semantic alignment}\rightarrow M\rightarrow\text{TPI}
\end{cases}
}
$$

This keeps ASR separate from the actual content-preservation evaluation.

### Next step

Do **not** choose an anonymization route inside this notebook. First establish the paired-data interface; then choose the appropriate route and run the first real paired experiment.

In [11]:
print("Notebook ready.")
print("Next required input: a dataframe of original/anonymized audio pairs.")


Notebook ready.
Next required input: a dataframe of original/anonymized audio pairs.
